# Component 3: Explainable AI and Severity: training and results notebook
**Project J-26-CS-314** | S.M.K.G.M.M. Sasangith (IT23348066)

This notebook trains and evaluates both machine-learning models used in Component 3, then shows SHAP explanations, severity decisions and processing time:
- **Random Forest classifier**: stand-in attack detector (18 attack classes) on CICIoMT2024. Component 2's real model will replace it.
- **Decision Tree classifier**: optional ML severity model (Critical, High, Medium, Low).

## Cell 1: Import Libraries

In [ ]:
import os, sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# EDIT these paths if your folders are different
PROJECT   = os.environ.get("C3_NB_PROJECT",   r"C:\Users\Menuka\Desktop\j26-CS-314\c3_xai_severity")
TRAIN_CSV = os.environ.get("C3_NB_TRAIN_CSV", r"C:\Users\Menuka\Desktop\dataset\archive (1)\train_iomt.csv")
TEST_CSV  = os.environ.get("C3_NB_TEST_CSV",  r"C:\Users\Menuka\Desktop\dataset\archive (1)\test_iomt.csv")

SAVE_MODELS = True          # True: overwrite models/detector.joblib and models/severity.joblib
TRAIN_ROWS_PER_CLASS = 20000
ESCALATE_CONF = 0.97        # severity policy: High becomes Critical when confidence >= this (agree with supervisor)

os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

from src import config, taxonomy
from src.contract_loader import FEATURES, DEVICES
from src.explain import Explainer
from src.severity import SeverityEngine, ml_features
from train_standin import load_big_csv

print("Project folder:", os.getcwd())
print("Contract features:", len(FEATURES))

## Cell 2: Load Training Dataset
The training file is very large (1.6 GB), so it is read in chunks and at most 20,000 rows per attack class are kept. Short dataset labels (e.g. `DDoS SYN`) are mapped to the contract class names. This cell takes a few minutes.

In [ ]:
train = load_big_csv(TRAIN_CSV, "label", per_class=TRAIN_ROWS_PER_CLASS)
print("\nTraining rows loaded:", len(train))

## Cell 3: Preprocessing
Remove infinite or missing values, then split into the 45 contract features (X) and the attack label (y).

In [ ]:
before = len(train)
train = train.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES)
X_train, y_train = train[FEATURES], train["label"]
print(f"Rows before: {before} | after cleaning: {len(train)} | removed: {before - len(train)}")
print("Feature matrix:", X_train.shape)

## Cell 4: Class Distribution (training set)

In [ ]:
counts = y_train.value_counts().sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(counts.index, counts.values, color="#2e86c1")
for y, v in enumerate(counts.values):
    ax.text(v + 150, y, str(v), va="center", fontsize=8)
ax.set_xlabel("rows"); ax.set_title(f"Training set: rows per attack class (max {TRAIN_ROWS_PER_CLASS:,} each)")
plt.tight_layout(); plt.show()

## Cell 5: Train Random Forest (stand-in detector)
Multi-class classification: the model predicts which of the attack classes a network flow belongs to.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

t0 = time.time()
model = RandomForestClassifier(n_estimators=60, max_depth=14, n_jobs=-1, random_state=42)
model.fit(X_train, y_train)
print(f"Random Forest trained in {time.time() - t0:.0f} s | classes: {len(model.classes_)} | trees: {model.n_estimators}")

if SAVE_MODELS:
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, config.MODEL_PATH)
    print("saved", config.MODEL_PATH)

## Cell 6: Load Test Dataset
At most 2,000 rows per attack class from the separate test file.

In [ ]:
test = load_big_csv(TEST_CSV, "label", per_class=2000)
test = test.replace([np.inf, -np.inf], np.nan).dropna(subset=FEATURES)
X_test, y_test = test[FEATURES], test["label"]
print("\nTest rows:", len(test))

## Cell 7: Random Forest Accuracy and Classification Report

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
macro_f1 = f1_score(y_test, pred, average="macro", zero_division=0)
print(f"Accuracy: {acc:.4f} | Macro F1: {macro_f1:.4f}\n")
print(classification_report(y_test, pred, zero_division=0))

## Cell 8: Confusion Matrix
Rows = true attack, columns = predicted attack. A bright diagonal means correct. Numbers are row percentages.

In [ ]:
from sklearn.metrics import confusion_matrix
labels = sorted(set(y_test) | set(pred))
cm = confusion_matrix(y_test, pred, labels=labels)
cm_pct = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(labels))); ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=90, fontsize=8); ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Confusion matrix (row %)")
for i in range(len(labels)):
    for j in range(len(labels)):
        if cm_pct[i, j] >= 0.05:
            ax.text(j, i, f"{cm_pct[i, j]*100:.0f}", ha="center", va="center", fontsize=7,
                    color="white" if cm_pct[i, j] > 0.5 else "black")
fig.colorbar(im, ax=ax, fraction=0.04)
plt.tight_layout(); plt.show()

## Cell 9: F1-score per Attack Class
Red bars are below 0.8. These weak classes belong to the stand-in detector (Component 2's job), not to Component 3.

In [ ]:
f1 = pd.Series(f1_score(y_test, pred, labels=labels, average=None, zero_division=0), index=labels).sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(f1.index, f1.values, color=["#c0392b" if v < 0.8 else "#2e86c1" for v in f1.values])
ax.set_xlim(0, 1); ax.set_xlabel("F1-score"); ax.set_title("F1 per class (red = below 0.8)")
for y, v in enumerate(f1.values):
    ax.text(v + 0.01, y, f"{v:.2f}", va="center", fontsize=8)
plt.tight_layout(); plt.show()

## Cell 10: SHAP Explanation for One Attack
SHAP shows which features pushed the model toward its decision. Blue pushes toward the predicted attack, red pushes away. Component 3 sends the top 3 to Component 4.

In [ ]:
explainer = Explainer(model, top_k=10)
attack = "TCP_IP-DDoS-SYN" if "TCP_IP-DDoS-SYN" in set(y_test) else y_test.iloc[0]
row = test[test["label"] == attack].iloc[0]
x = [float(row[c]) for c in FEATURES]
top, total = explainer.analyse(x, attack)

names  = [t["feature"] for t in top][::-1]
values = [t["shap_value"] for t in top][::-1]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(names, values, color=["#2e86c1" if v > 0 else "#c0392b" for v in values])
ax.axvline(0, color="black", lw=0.8)
ax.set_title(f"Top 10 SHAP features for a '{attack}' flow"); ax.set_xlabel("SHAP value")
plt.tight_layout(); plt.show()
print("Top 3 sent to Component 4:", top[:3])

## Cell 11: Train Decision Tree (ML severity model)
Predicts the severity (Critical, High, Medium, Low) from: attack type, detector confidence, SHAP magnitude, probability margin and the life-critical device flag.

**Honest note:** the severity labels come from the attack table plus the life-critical rule, so a model trained only on attack type would just re-learn that table. The label policy therefore includes a continuous rule (`ESCALATE_CONF`): High becomes Critical when the detector confidence is at or above the threshold. This policy must be agreed with the supervisor. The rule-only baseline below does not know about the confidence rule, so the comparison shows what the tree adds.

In [ ]:
import random
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

random.seed(0)
explainer3 = Explainer(model, top_k=3)
pool = train[train["label"].isin(taxonomy.ATTACK_SEVERITY) & train["label"].isin(explainer3.classes)]
pool = pool.sample(min(1000, len(pool)), random_state=0)          # SHAP is slow, 1000 rows is enough

proba_all = model.predict_proba(pool[FEATURES])
devices = list(DEVICES)
rows, labels_sev, meta = [], [], []
for i, (_, r) in enumerate(pool.iterrows()):
    cls = r["label"]
    proba = proba_all[i]
    order = np.argsort(-proba)
    conf = float(proba[explainer3.classes.index(cls)])
    margin = float(proba[order[0]] - proba[order[1]])
    _, mag = explainer3.analyse([float(r[c]) for c in FEATURES], cls)
    dev = random.choice(devices)
    lc = taxonomy.device_is_life_critical(dev)
    rows.append(ml_features(cls, conf, mag, margin, lc).iloc[0])
    labels_sev.append(taxonomy.apply_override(taxonomy.rule_severity(cls, conf, ESCALATE_CONF), lc)[0])
    meta.append((cls, lc))

X_sev, y_sev = pd.DataFrame(rows), pd.Series(labels_sev)
idx = np.arange(len(y_sev))
strat = y_sev if y_sev.value_counts().min() >= 2 else None
tr, te = train_test_split(idx, test_size=0.2, random_state=42, stratify=strat)

tree = DecisionTreeClassifier(max_depth=6, random_state=42).fit(X_sev.iloc[tr], y_sev.iloc[tr])
tree_pred = tree.predict(X_sev.iloc[te])
base_pred = [taxonomy.apply_override(taxonomy.rule_severity(meta[i][0], None, None), meta[i][1])[0] for i in te]

print("Severity label distribution:\n", y_sev.value_counts().to_string(), "\n")
print(f"Rule-only baseline accuracy (no confidence rule): {np.mean(np.array(base_pred) == y_sev.iloc[te].values):.3f}")
print(f"Decision Tree accuracy:                           {np.mean(tree_pred == y_sev.iloc[te].values):.3f}\n")
print(classification_report(y_sev.iloc[te], tree_pred, zero_division=0))

if SAVE_MODELS:
    joblib.dump(tree, config.SEVERITY_MODEL_PATH)
    print("saved", config.SEVERITY_MODEL_PATH, "| run the service with C3_SEVERITY_MODE=ml to use it")

## Cell 12: Severity Decisions and the Life-Critical Override
Same attack, different devices (rule-based severity). Life-critical devices are forced to **Critical** with the protection flag, whatever the attack.

In [ ]:
engine = SeverityEngine(mode="rule")
attacks = ["Recon-Ping_Sweep", "Recon-OS_Scan", "TCP_IP-DoS-SYN", "TCP_IP-DDoS-SYN"]
devices = list(DEVICES)
life = [d for d in devices if taxonomy.device_is_life_critical(d)]
print("Life-critical devices:", life, "\n")

pd.DataFrame(
    {dev: [engine.decide(a, 0.90, 0.0, dev)[0] for a in attacks] for dev in devices},
    index=attacks,
)

## Cell 13: Processing Time
Time Component 3 needs to explain one detection (SHAP) and decide its severity. Target: under 100 ms.

In [ ]:
sample = test.sample(min(200, len(test)), random_state=0)
explainer3.analyse([float(sample.iloc[0][c]) for c in FEATURES], sample.iloc[0]["label"])   # warm-up

times = []
for _, r in sample.iterrows():
    cls = r["label"]
    if cls not in taxonomy.ATTACK_SEVERITY or cls not in explainer3.classes:
        continue
    x = [float(r[c]) for c in FEATURES]
    t0 = time.perf_counter()
    top, mag = explainer3.analyse(x, cls)
    engine.decide(cls, 0.9, mag, "bp_monitor_01")
    times.append((time.perf_counter() - t0) * 1000)

times = np.array(times)
print(f"requests: {len(times)} | median: {np.median(times):.1f} ms | p90: {np.percentile(times, 90):.1f} ms | max: {times.max():.1f} ms")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(times, bins=30, color="#2e86c1"); ax.axvline(100, color="#c0392b", ls="--", label="100 ms target")
ax.set_xlabel("ms per detection"); ax.set_ylabel("count"); ax.legend(); ax.set_title("Component 3 processing time")
plt.tight_layout(); plt.show()

## Cell 14: Final Summary

In [ ]:
weak = f1.head(3)
summary = pd.DataFrame({
    "Item": [
        "Random Forest (stand-in detector) accuracy",
        "Random Forest macro F1",
        "Weakest classes (F1)",
        "Decision Tree severity accuracy (vs rule-only baseline)",
        "Contract features used",
        "Life-critical devices (forced Critical)",
        "SHAP + severity time (median / p90)",
        "Under 100 ms target",
    ],
    "Result": [
        f"{acc:.1%} on {len(test)} real test rows",
        f"{macro_f1:.3f}",
        ", ".join(f"{k} ({v:.2f})" for k, v in weak.items()),
        f"{np.mean(tree_pred == y_sev.iloc[te].values):.1%} (baseline {np.mean(np.array(base_pred) == y_sev.iloc[te].values):.1%})",
        str(len(FEATURES)),
        ", ".join(life),
        f"{np.median(times):.1f} ms / {np.percentile(times, 90):.1f} ms",
        "Yes" if np.percentile(times, 90) < 100 else "No",
    ],
})
print(summary.to_string(index=False))

### Notes
- The Random Forest is a **stand-in** detector. Component 2's real model will replace `models/detector.joblib`.
- Weak classes belong to the stand-in detector, not to Component 3.
- The severity table and the `ESCALATE_CONF` policy are not final until the supervisor approves them.